# Sampling (capsid) surfaces

In this notebook we load a segmented capsid volume and sample points on the surface(s). 

The workflow uses `PleomorphicSurface` as a wrapper. When methods are called on the wrapper (`read`, `refine_normals`, `oversample`, `separate_surfaces`, …); it forwards to the right surface implementation and, when an operation returns new geometry, wraps the result again so you stay in the same API. Under the hood, the surface is represented as one of two types:

- `Mesh`: Triangle surface with vertices, faces, and normals.
- `OrientedPointCloud`: Discrete points with normals.

Meshes and oriented point clouds share many methods (normal refinement, cropping, inner/outer separation, distance queries) but these can differ in their implementation. The representation of an object can also change with certain methods. `PleomorphicSurface` internally tracks these representational changes with a single notebook-facing object for both stages so you do not re-wrap after every step.

The wrapper exposes **`is_mesh`** and **`is_point_cloud`** so you can see which representation you have without inspecting types manually.

### Workflow

1. **Load** the segmentation as a mesh-backed `PleomorphicSurface`.
2. **Optional:** Split inner and outer capsid surfaces by normal orientation around a centroid.
3. **Oversample** each surface to an approximately uniform oriented point cloud.
4. **Save** meshes, motive lists, and point cloud objects.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import numpy as np
from cryocat.analysis import structure                              # local structure.py (imports local surface)

## Load the segmentation

The capsid surface is built with **`PleomorphicSurface.read`** and `method="mesh_from_mrc"`. Under the hood this runs marching cubes on the MRC volume and stores a **`Mesh`** in `capsid.surface`.

Marching cubes produces vertices, faces, and normals. Vertex coordinates are scaled by **`pixel_size`** during extraction:

- If pixel_size=1`, coordinates in pixel/voxel units.
- Known physical spacing, e.g. `pixel_size=0.5` (nm) or `pixel_size=5.0` (A) → vertices in those units.

The unit label is stored separately with `.units` and can be set after creating the mesh:
```python
capsid.units = "pixel"       # or "nm", "A"
```

The main other choices at this stage are whether to smooth the segmentation before meshing and whether to call `refine_normals()` after loading.

| Parameter | Typical value | Meaning |
|---|---:|---|
| `input_path` | MRC file path | Segmentation volume to load. |
| `transpose` | `True` | Uses the package convention for reading MRC volumes. Keep this consistent across the notebook. |
| `pixel_size` | `1` if no spacing is known | Coordinate spacing. If `1`, output coordinates are in pixel/voxel units. If physical pixel size is known, use that value instead. |
| `smooth_sigma` | `None` to `2.5` | Gaussian smoothing applied before marching cubes, in pixel/voxel units. Larger values reduce blocky artifacts but can also soften details. |
| `level` | `0.5` | Iso-value for marching cubes. For binary masks, `0.5` is usually appropriate. |

In [3]:
# Input segmentation path
input_path = "inputs/"
capsid_seg = input_path + "capsid_mask.mrc"

In [4]:
capsid = structure.PleomorphicSurface.read(
    input_path=capsid_seg,
    method="mesh_from_mrc",
    transpose=True,
    pixel_size=1,
    smooth_sigma=2.0,
    level=0.5,
)

# Optional: after marching cubes; tune via radius_hit / n_iter on the method
capsid = capsid.refine_normals(radius_hit=3.0, n_iter=1)

Applying Gaussian smoothing (sigma=2.0)...
Smoothed field range: [0.000, 1.000]
Running marching cubes (level=0.5, step_size=1)...
Refining normals (radius=3.0, batch=2000, iter=1, mask=all)
Iteration 1/1


100%|██████████| 243/243 [00:34<00:00,  6.99it/s]

Normal refinement complete.


In [5]:
capsid.units = "pixel"

In [6]:
# Check internal representation
print(f"capsid.is_mesh: {capsid.is_mesh}")
print(f"capsid.is_point_cloud: {capsid.is_point_cloud}")

capsid.is_mesh: True
capsid.is_point_cloud: False


## Optional: Separate inner and outer capsid surfaces

For a "hollow" capsid segmentation, the expected convention after marching cubes is that the outer surface normals point away from the capsid center, while the inner surface normals point toward the capsid center. During the inner/outer separation each normal tis compared o the direction from the vertex toward a reference point, usually the centroid. Vertices whose normals point toward the reference are assigned to the inner surface; the rest to the outer surface.

`separate_surfaces` dispatches to the appropriate separation algorithm based on `surface_type` and returns two boolean masks without modifying the geometry. For enclosed geometries such as capsids, use `surface_type='closed'`. For flatter surfaces where inner/outer distinction is not meaningful, use `surface_type='planar'` instead.

| Parameter | Default | Meaning |
|---|---:|---|
| `surface_type` | `'closed'` | Separation strategy. `'closed'` for enclosed volumes (capsids, vesicles); `'planar'` for surfaces with lower curvature. |
| `threshold_angle` | `90.0` | Angle cutoff in degrees. Smaller values make the inner assignment stricter. Only used when `surface_type='closed'`. |
| `reference_point` | `None` | Reference point for classification. If `None`, the centroid is used. For asymmetric objects, provide a point known to lie inside the capsid. Only used when `surface_type='closed'`. |

This approach works best for closed or nearly closed nested surfaces where normals are consistently oriented. If the normals are noisy or the shape is strongly open or asymmetric, inspect the result before continuing.

In [7]:
inner_mask, outer_mask = capsid.separate_surfaces(surface_type='closed')

### Extract the inner and outer surfaces

`apply_vertex_mask` filters the surface to only the vertices where the mask is `True`, without modifying the original object. Pass `inner_mask` to get the inner shell and `outer_mask` to get the outer shell.

| Mask | Meaning |
|---|---|
| `inner_mask` | Vertices whose normals point toward the centroid (inner capsid shell). |
| `outer_mask` | Vertices whose normals point away from the centroid (outer capsid shell). |

The result keeps the same representation as the input:

- If the input is a mesh, the output is a mesh containing only the selected vertices and valid faces.
- If the input is an oriented point cloud, the output is an oriented point cloud containing only the selected points and normals.

For this workflow, `inner_surface` and `outer_surface` should still be mesh-backed `PleomorphicSurface` objects after filtering.

In [8]:
inner_surface = capsid.apply_vertex_mask(inner_mask)
outer_surface = capsid.apply_vertex_mask(outer_mask)

In [9]:
print(f"inner_surface.is_mesh: {inner_surface.is_mesh}")
print(f"outer_surface.is_mesh: {outer_surface.is_mesh}")
print(f"inner_surface.is_point_cloud: {inner_surface.is_point_cloud}")
print(f"outer_surface.is_point_cloud: {outer_surface.is_point_cloud}")
print(f"Number of points in inner_surface: {len(inner_surface.surface.vertices)}")
print(f"Number of points in outer_surface: {len(outer_surface.surface.vertices)}")

inner_surface.is_mesh: True
outer_surface.is_mesh: True
inner_surface.is_point_cloud: False
outer_surface.is_point_cloud: False
Number of points in inner_surface: 191356
Number of points in outer_surface: 293086


### Report mesh surface area
If the input is a mesh, one can report the surface area (`PleomorphicSurface` calls the method on `Mesh`).
If the input is an `OrientedPointCloud` a `NotImplementedError` will be raised, as an area on a point cloud is undefined without triangle connectivity.

In [10]:
print(f"inner_surface.get_surface_area(): {inner_surface.get_surface_area().round(0)} (pixels^2)")
print(f"outer_surface.get_surface_area(): {outer_surface.get_surface_area().round(0)} (pixels^2)")

inner_surface.get_surface_area(): 135340.0 (pixels^2)
outer_surface.get_surface_area(): 204423.0 (pixels^2)


## Oversample the mesh surface

Oversampling converts each mesh surface into a more uniformly sampled oriented point cloud.

When the starting object is a `Mesh`, the implementation uses mesh-based Poisson disk sampling. The code first estimates a target number of points from either `point_spacing` or `oversample_factor`. If the target number of points is larger than the current number of mesh vertices, the mesh may be subdivided internally before sampling. This gives the sampler enough geometric detail to place points more evenly on the surface.

| Parameter | Typical use | Meaning |
|---|---:|---|
| `point_spacing` | `8.0` here | Desired approximate distance between sampled points, in the same units as the mesh vertices. |
| `oversample_factor` | optional | Alternative to `point_spacing`; target point count is based on the current number of vertices. |
| `poisson_init_factor` | `5` | Controls the candidate pool used by Poisson disk sampling. Larger values can improve uniformity but cost more time. |

After mesh oversampling, the wrapped object is no longer a mesh. It becomes a `PleomorphicSurface` backed by an `OrientedPointCloud`, because the output is sampled points with normals rather than connected faces.

In [11]:
spacing = 8.0  # same units as mesh vertices (voxels here)

inner_surface_oversampled = inner_surface.oversample(point_spacing=spacing)
outer_surface_oversampled = outer_surface.oversample(point_spacing=spacing)

Target points: 1529 (0.01x original)
Final: 1529 points
Target points: 2300 (0.01x original)
Final: 2300 points


In [12]:
print(f"inner_surface_oversampled.is_point_cloud: {inner_surface_oversampled.is_point_cloud}")
print(f"outer_surface_oversampled.is_point_cloud: {outer_surface_oversampled.is_point_cloud}")
print(f"Number of points in inner_surface_oversampled: {len(inner_surface_oversampled.surface.vertices)}")
print(f"Number of points in outer_surface_oversampled: {len(outer_surface_oversampled.surface.vertices)}")

inner_surface_oversampled.is_point_cloud: True
outer_surface_oversampled.is_point_cloud: True
Number of points in inner_surface_oversampled: 1529
Number of points in outer_surface_oversampled: 2300


## Oversample point clouds

If `oversample` is called on a `PleomorphicSurface` that represents an `OrientedPointCloud`, the implementation is somewhat different.

A point cloud has no faces and no continuous mesh surface. Therefore the code cannot subdivide triangles or use mesh-based Poisson disk sampling. Instead, it uses a KD-tree and a greedy spacing-based selection on the existing points.

This means:

- Mesh input: sample over a continuous surface, with optional internal subdivision.
- Mesh oversampling usually changes the representation from `Mesh` to `OrientedPointCloud`.
- Point-cloud input: operate directly on existing points.
- Point-cloud oversampling keeps the representation as `OrientedPointCloud`.

In [13]:
# Create a point cloud from the capsid mesh
# Or load a point cloud from a motive list or .ply file
capsid_pc = capsid.surface.to_oriented_points()

In [14]:
# Wrap the point cloud as a PleomorphicSurface
capsid_pc = structure.PleomorphicSurface(capsid_pc)
print(f"capsid_pc.is_point_cloud: {capsid_pc.is_point_cloud}")

capsid_pc.is_point_cloud: True


In [15]:
# The same approach for using a centroid to separate the surfaces can be applied to the point cloud
inner_mask_pc, outer_mask_pc = capsid_pc.separate_surfaces(surface_type='closed')

In [16]:
# Generate two separate point clouds corresponding to the inner and outer surfaces
inner_surface_pc = capsid_pc.apply_vertex_mask(inner_mask_pc)
outer_surface_pc = capsid_pc.apply_vertex_mask(outer_mask_pc)

In [17]:
spacing = 8.0

inner_surface_oversampled_pc = inner_surface_pc.oversample(point_spacing=spacing)
outer_surface_oversampled_pc = outer_surface_pc.oversample(point_spacing=spacing)

Point cloud sampling: 191356 points
  Current spacing: 0.51, target spacing: 8.00
Final: 1423 points
Point cloud sampling: 293086 points
  Current spacing: 0.50, target spacing: 8.00
Final: 2133 points


In [18]:
print(f"inner_surface_oversampled_pc.is_point_cloud: {inner_surface_oversampled_pc.is_point_cloud}")
print(f"outer_surface_oversampled_pc.is_point_cloud: {outer_surface_oversampled_pc.is_point_cloud}")
print(f"Number of points in inner_surface_oversampled_pc: {len(inner_surface_oversampled_pc.surface.vertices)}")
print(f"Number of points in outer_surface_oversampled_pc: {len(outer_surface_oversampled_pc.surface.vertices)}")

inner_surface_oversampled_pc.is_point_cloud: True
outer_surface_oversampled_pc.is_point_cloud: True
Number of points in inner_surface_oversampled_pc: 1423
Number of points in outer_surface_oversampled_pc: 2133


## Saving results

Saving is done with a general `.save` method implemented in `PleomorphicSurface`. The two underlying representations have different natural output formats:

| Object type | Typical output | Method |
|---|---|---|
| `Mesh` | `.ply` \ `.vtp` mesh | `save(..., format="ply" \ "vtp")` |
| `OrientedPointCloud` | motive list `.em` | `save(..., tomo_id=[], subtomo_ids=[])` |
| `OrientedPointCloud` | point-cloud `.ply` | `save(..., format="ply")` |

In this tutorial, the inner and outer capsid surfaces are meshes that we save as `.ply` or `.vtp` files After oversampling, the resulting points are an `OrientedPointCloud` object, that we can export as motive lists or a `.ply` point cloud.

For motive-list export, two additional parameters can be set:

| Parameter | Example |
|---|---|
| `tomo_id` | Pass a scalar such as `1` when all sampled points come from the same tomogram (numbered one), or an array if points come from multiple tomograms. |
| `subtomo_ids` | Object or particle grouping for each point. The array must have one value per sampled point. If all points belong to one object, `np.ones(n_points)`. If you pass different object labels, the method maps unique labels to sequential `subtomo_id` values. |

In [19]:
# Before saving, check the representation
print(f"inner_surface.is_mesh: {inner_surface.is_mesh}")
print(f"outer_surface.is_mesh: {outer_surface.is_mesh}")
print(f"inner_surface_oversampled.is_point_cloud: {inner_surface_oversampled.is_point_cloud}")
print(f"outer_surface_oversampled.is_point_cloud: {outer_surface_oversampled.is_point_cloud}")


inner_surface.is_mesh: True
outer_surface.is_mesh: True
inner_surface_oversampled.is_point_cloud: True
outer_surface_oversampled.is_point_cloud: True


In [20]:
output_path = "outputs/"
os.makedirs(output_path, exist_ok=True)

In [21]:
# Save the filtered mesh surfaces
inner_surface.save(
    output_path + "capsid_inner_surface.ply", 
    format='ply', 
    include_curvatures=False)

inner_surface.save(
    output_path + "capsid_inner_surface.vtp", 
    format='vtp', 
    include_curvatures=False)

outer_surface.save(
    output_path + "capsid_outer_surface.ply", 
    format='ply', 
    include_curvatures=False)

outer_surface.save(
    output_path + "capsid_outer_surface.vtp", 
    format='vtp', 
    include_curvatures=False)


Saved mesh to outputs/capsid_inner_surface.ply
  Vertices: 191,356
  Faces: 382,455
Saved mesh to outputs/capsid_inner_surface.vtp
  Format: VTP (VTK PolyData)
  Vertices: 191,356
  Faces: 382,455
Saved mesh to outputs/capsid_outer_surface.ply
  Vertices: 293,086
  Faces: 585,922
Saved mesh to outputs/capsid_outer_surface.vtp
  Format: VTP (VTK PolyData)
  Vertices: 293,086
  Faces: 585,922


- Mesh of the outer capsid surface
<img src="media/capsid_outer_surface_mesh.png" width="800"/>
- Mesh of the inner capsid surface
<img src="media/capsid_inner_surface_mesh.png" width="800"/>
- Both meshes overlaid
<img src="media/capsid_both_surfaces_mesh.png" width="800"/>

In [22]:
# Save the oversampled point clouds as motive lists
inner_surface_oversampled.save(
    output_path + "capsid_inner_surface_sampled_points.em",
    tomo_id=1,
    subtomo_ids=np.ones(len(inner_surface_oversampled.surface.vertices))
)

outer_surface_oversampled.save(
    output_path + "capsid_outer_surface_sampled_points.em",
    tomo_id=1,
    subtomo_ids=np.ones(len(outer_surface_oversampled.surface.vertices))
)

Saved 1529 points with normals to outputs/capsid_inner_surface_sampled_points.em
Saved 2300 points with normals to outputs/capsid_outer_surface_sampled_points.em


In [23]:
# Save the oversampled point clouds as point clouds
inner_surface_oversampled.save(
    output_path + "capsid_inner_surface_sampled_points.ply",
    format='ply',
)

outer_surface_oversampled.save(
    output_path + "capsid_outer_surface_sampled_points.ply",
    format='ply',
)

Saved point cloud with normals to outputs/capsid_inner_surface_sampled_points.ply
Saved point cloud with normals to outputs/capsid_outer_surface_sampled_points.ply


- Motive list of the outer surface (including orientations)
<img src="media/capsid_outer_surface_motl.png" width="800"/>

- Motive list of the inner surface (including orientations)
<img src="media/capsid_inner_surface_motl.png" width="800"/>

- Both motive lists overlaid
<img src="media/capsid_both_surfaces_motl.png" width="800"/>

In [24]:
# If the normals should point in the opposite direction, flip them
inner_surface_oversampled_fipped = inner_surface_oversampled.flip_normals()

In [25]:
inner_surface_oversampled_fipped.save(
    output_path + "capsid_inner_surface_sampled_points_flipped.em",
    tomo_id=1,
    subtomo_ids=np.ones(len(inner_surface_oversampled.surface.vertices))
)

Saved 1529 points with normals to outputs/capsid_inner_surface_sampled_points_flipped.em
